In [ ]:
# ============================================================
# STEP 1: Install Dependencies
# Run once. Restart kernel after installation.
# ============================================================
# !pip install openai faiss-cpu mysql-connector-python fastapi uvicorn \
#              python-multipart nltk tiktoken numpy pandas \
#              httpx python-dotenv nest-asyncio requests

In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
import os, re, json, time
import numpy as np
import pandas as pd
import faiss
import mysql.connector
import nltk
import warnings
warnings.filterwarnings('ignore')

from openai import OpenAI
from nltk.tokenize import sent_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("All libraries imported successfully!")

In [ ]:
# ============================================================
# STEP 3: Configuration
# IMPORTANT: Replace the placeholders below with your real credentials
# ============================================================

OPENAI_API_KEY = "sk-your-openai-api-key-here"   # <-- REPLACE

MYSQL_CONFIG = {
    "host"    : "localhost",
    "port"    : 3306,
    "user"    : "root",                           # <-- REPLACE if different
    "password": "your_mysql_password",            # <-- REPLACE
    "database": "contact_center_rag"
}

EMBEDDING_MODEL = "text-embedding-ada-002"
CHAT_MODEL      = "gpt-3.5-turbo"                # change to "gpt-4" if available
EMBEDDING_DIM   = 1536
TOP_K           = 3

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"Config ready. Chat model: {CHAT_MODEL}")

In [ ]:
# ============================================================
# STEP 4: Sample Contact Center Documents
# In a real project, load from PDFs/Word docs using PyPDF2 or python-docx
# ============================================================

DOCUMENTS = [
    {
        "doc_id"  : "DOC001",
        "doc_name": "Refund and Return Policy",
        "category": "refund",
        "content" : """
        Refund and Return Policy
        Our company offers a 30-day return policy for all products. Customers can return
        any product within 30 days of purchase for a full refund, provided the item is
        in its original condition with all tags and packaging intact.
        To initiate a return, customers must contact our support team via email at
        returns@company.com or call our helpline at 1800-123-4567. A Return Merchandise
        Authorization (RMA) number will be provided within 24 hours.
        Refunds are processed within 5-7 business days after we receive the returned item.
        The refund will be credited to the original payment method. For credit card payments,
        the refund may take an additional 3-5 business days to reflect on your statement.
        Non-refundable items include: digital downloads, personalized products, perishable
        goods, and items marked as final sale. Gift cards cannot be returned or refunded.
        For defective products, we offer a replacement or full refund within 90 days of
        purchase. Customers must provide photo evidence of the defect when contacting support.
        If a return is requested after 30 days but within 60 days, we offer store credit
        equivalent to the purchase price. No returns are accepted after 60 days.
        """
    },
    {
        "doc_id"  : "DOC002",
        "doc_name": "Shipping and Delivery FAQ",
        "category": "shipping",
        "content" : """
        Shipping and Delivery Frequently Asked Questions
        Q: How long does standard shipping take?
        A: Standard shipping typically takes 5-7 business days within the country.
        International shipping takes 10-15 business days depending on the destination country.
        Q: Do you offer express delivery?
        A: Yes, we offer express delivery (1-2 business days) for an additional fee.
        Express delivery is available for orders placed before 12 PM local time.
        Same-day delivery is available in select cities for orders placed before 10 AM.
        Q: How can I track my order?
        A: Once your order is shipped, you will receive a tracking number via email and SMS.
        You can track your order on our website under 'My Orders' or directly on the
        courier's website. Tracking updates are available every 4-6 hours.
        Q: What happens if my package is lost?
        A: If your package is not delivered within 15 business days of the estimated
        delivery date, please contact our support team. We will initiate a trace with
        the courier. If the package is confirmed lost, we will reship the order at no
        additional cost or provide a full refund.
        Q: Are there free shipping options?
        A: Free standard shipping is available on all orders above Rs. 999.
        Members of our loyalty program get free shipping on all orders.
        """
    },
    {
        "doc_id"  : "DOC003",
        "doc_name": "Account Management Guide",
        "category": "account",
        "content" : """
        Customer Account Management Guide
        Creating an Account:
        To create an account, visit our website and click on 'Sign Up'. Enter your email
        address, create a password (minimum 8 characters with at least one uppercase letter,
        one number, and one special character), and provide your name and mobile number.
        A verification email will be sent. Click the link in the email to activate your account.
        Resetting Your Password:
        If you forget your password, click 'Forgot Password' on the login page. Enter your
        registered email address and you will receive a password reset link within 5 minutes.
        The reset link expires after 30 minutes. If you don't receive the email, check your
        spam folder or contact support@company.com.
        Updating Account Information:
        You can update your name, phone number, and address from 'My Profile' settings.
        To change your email address, you must verify the new email first. Changes to
        payment methods can be made under 'Payment Settings'.
        Two-Factor Authentication (2FA):
        We strongly recommend enabling 2FA for added security. Go to Settings > Security >
        Enable 2FA. You can use an authenticator app (Google Authenticator, Authy) or
        receive OTP via SMS. If you lose access to your 2FA method, contact our support
        team with your account email and a government-issued ID for verification.
        """
    }
]

print(f"Loaded {len(DOCUMENTS)} documents")
for d in DOCUMENTS:
    print(f"  [{d['doc_id']}] {d['doc_name']}")

In [ ]:
# ============================================================
# STEP 5: Text Cleaning and Chunking
# ============================================================

def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return text

def chunk_text(text: str, chunk_size: int = 150, overlap: int = 30) -> list:
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        end   = start + chunk_size
        chunk = ' '.join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Process all documents into chunks
all_chunks = []
for doc in DOCUMENTS:
    cleaned = clean_text(doc['content'])
    chunks  = chunk_text(cleaned)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "chunk_id"   : f"{doc['doc_id']}_C{i+1:03d}",
            "doc_id"     : doc['doc_id'],
            "doc_name"   : doc['doc_name'],
            "category"   : doc['category'],
            "chunk_index": i,
            "chunk_text" : chunk,
            "word_count" : len(chunk.split())
        })

df_chunks = pd.DataFrame(all_chunks)
print(f"Total chunks created: {len(all_chunks)}")
print(df_chunks.groupby('doc_name')['chunk_id'].count())

In [ ]:
# ============================================================
# STEP 6: Generate OpenAI Embeddings
# Note: This makes API calls — may take 1-2 minutes
# ============================================================

def get_embedding(text: str) -> list:
    response = client.embeddings.create(
        input=[text.replace('\n', ' ')],
        model=EMBEDDING_MODEL
    )
    return response.data[0].embedding

def get_embeddings_batch(texts: list, batch_size: int = 20) -> list:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch    = [t.replace('\n', ' ') for t in texts[i:i+batch_size]]
        response = client.embeddings.create(input=batch, model=EMBEDDING_MODEL)
        sorted_e = sorted(response.data, key=lambda x: x.index)
        all_embeddings.extend([e.embedding for e in sorted_e])
        print(f"  Embedded {min(i+batch_size, len(texts))}/{len(texts)} chunks...")
        time.sleep(0.5)
    return all_embeddings

chunk_texts      = [c['chunk_text'] for c in all_chunks]
print(f"Generating embeddings for {len(chunk_texts)} chunks...")
embeddings       = get_embeddings_batch(chunk_texts)
embeddings_array = np.array(embeddings, dtype='float32')

print(f"Embeddings shape: {embeddings_array.shape}")

In [ ]:
# ============================================================
# STEP 7: Build and Save FAISS Index
# ============================================================

def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)        # Inner Product = cosine similarity (after L2 norm)
    faiss.normalize_L2(embeddings)        # Normalize vectors for cosine similarity
    index.add(embeddings)
    return index

faiss_index = build_faiss_index(embeddings_array.copy())

# Save index and metadata to disk
faiss.write_index(faiss_index, "faiss_index.bin")
with open("chunks_metadata.json", 'w') as f:
    json.dump(all_chunks, f, indent=2)

print(f"FAISS index built: {faiss_index.ntotal} vectors")
print("Saved: faiss_index.bin, chunks_metadata.json")

In [ ]:
# ============================================================
# STEP 8: FAISS Similarity Search Function
# ============================================================

def search_faiss(query: str, index: faiss.Index, chunks: list, top_k: int = TOP_K) -> list:
    query_embedding = np.array([get_embedding(query)], dtype='float32')
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, top_k)
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        if idx == -1:
            continue
        result = chunks[idx].copy()
        result['similarity_score'] = float(dist)
        result['rank']             = rank + 1
        results.append(result)
    return results

# Quick test
test_q   = "How many days do I have to return a product?"
test_res = search_faiss(test_q, faiss_index, all_chunks)
print(f"Query: '{test_q}'")
for r in test_res:
    print(f"  Rank {r['rank']} | Score: {r['similarity_score']:.4f} | {r['doc_name']}")

In [ ]:
# ============================================================
# STEP 9: MySQL Database Setup
# Make sure MySQL server is running before executing this cell
# ============================================================

def get_db_connection(include_db: bool = True):
    config = MYSQL_CONFIG.copy()
    if not include_db:
        config.pop('database', None)
    return mysql.connector.connect(**config)

def setup_database():
    conn   = get_db_connection(include_db=False)
    cursor = conn.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {MYSQL_CONFIG['database']}")
    cursor.execute(f"USE {MYSQL_CONFIG['database']}")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS documents (
            doc_id       VARCHAR(20)  PRIMARY KEY,
            doc_name     VARCHAR(200) NOT NULL,
            category     VARCHAR(50),
            total_chunks INT DEFAULT 0,
            created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS document_chunks (
            chunk_id    VARCHAR(30)  PRIMARY KEY,
            doc_id      VARCHAR(20)  NOT NULL,
            doc_name    VARCHAR(200),
            category    VARCHAR(50),
            chunk_index INT,
            chunk_text  TEXT         NOT NULL,
            word_count  INT,
            faiss_index INT,
            created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (doc_id) REFERENCES documents(doc_id)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS query_logs (
            id           INT AUTO_INCREMENT PRIMARY KEY,
            query_text   TEXT         NOT NULL,
            top_chunk_id VARCHAR(30),
            answer_text  TEXT,
            response_ms  INT,
            queried_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.commit()
    cursor.close()
    conn.close()
    print("Database and tables created successfully")

setup_database()

In [ ]:
# ============================================================
# STEP 10: Insert Documents and Chunks into MySQL
# ============================================================

def insert_documents_and_chunks(documents: list, chunks: list):
    conn   = get_db_connection()
    cursor = conn.cursor()

    for doc in documents:
        count = len([c for c in chunks if c['doc_id'] == doc['doc_id']])
        cursor.execute(
            "INSERT IGNORE INTO documents (doc_id, doc_name, category, total_chunks) VALUES (%s, %s, %s, %s)",
            (doc['doc_id'], doc['doc_name'], doc['category'], count)
        )

    for faiss_idx, chunk in enumerate(chunks):
        cursor.execute("""
            INSERT IGNORE INTO document_chunks
            (chunk_id, doc_id, doc_name, category, chunk_index, chunk_text, word_count, faiss_index)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            chunk['chunk_id'], chunk['doc_id'], chunk['doc_name'], chunk['category'],
            chunk['chunk_index'], chunk['chunk_text'], chunk['word_count'], faiss_idx
        ))

    conn.commit()
    cursor.close()
    conn.close()
    print(f"Inserted {len(documents)} documents and {len(chunks)} chunks into MySQL")

insert_documents_and_chunks(DOCUMENTS, all_chunks)

# Verify
conn = get_db_connection()
df   = pd.read_sql("SELECT doc_id, doc_name, total_chunks FROM documents", conn)
conn.close()
print(df.to_string(index=False))

In [ ]:
# ============================================================
# STEP 11: Query Logging Function
# ============================================================

def log_query(query: str, top_chunk_id: str, answer: str, response_ms: int):
    try:
        conn   = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO query_logs (query_text, top_chunk_id, answer_text, response_ms) VALUES (%s, %s, %s, %s)",
            (query, top_chunk_id, answer[:2000], response_ms)
        )
        conn.commit()
        cursor.close()
        conn.close()
    except Exception as e:
        print(f"Logging error (non-critical): {e}")

print("log_query function ready")

In [ ]:
# ============================================================
# STEP 12: Build RAG Prompt
# ============================================================

def build_rag_prompt(question: str, retrieved_chunks: list) -> list:
    context_parts = []
    for chunk in retrieved_chunks:
        context_parts.append(
            f"[Document: {chunk['doc_name']} | Score: {chunk['similarity_score']:.3f}]\n"
            f"{chunk['chunk_text']}"
        )
    context = "\n\n".join(context_parts)

    system_prompt = """You are a helpful customer support AI assistant for a contact center.
Answer ONLY using information from the provided context documents.
If the answer is not in the context, say: 'I don't have information about that. Please contact our support team.'
Be concise, friendly, and professional. Do not make up information."""

    user_message = f"""Context Documents:
{context}

Customer Question: {question}

Answer:"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message}
    ]

print("build_rag_prompt function ready")

In [ ]:
# ============================================================
# STEP 13: Full RAG Query Pipeline
# ============================================================

def rag_query(question: str, log_to_db: bool = True) -> dict:
    start_time = time.time()

    # Step 1 - RETRIEVE
    retrieved_chunks = search_faiss(question, faiss_index, all_chunks, top_k=TOP_K)
    if not retrieved_chunks:
        return {"answer": "No relevant information found. Please contact support.", "sources": []}

    # Step 2 - AUGMENT
    messages = build_rag_prompt(question, retrieved_chunks)

    # Step 3 - GENERATE
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.2,
        max_tokens=500
    )
    answer      = response.choices[0].message.content.strip()
    response_ms = int((time.time() - start_time) * 1000)

    # Log to MySQL
    if log_to_db:
        log_query(question, retrieved_chunks[0]['chunk_id'], answer, response_ms)

    return {
        "answer" : answer,
        "sources": [
            {
                "chunk_id"  : r['chunk_id'],
                "doc_name"  : r['doc_name'],
                "category"  : r['category'],
                "similarity": round(r['similarity_score'], 4),
                "excerpt"   : r['chunk_text'][:200] + "..."
            } for r in retrieved_chunks
        ],
        "metadata": {"model": CHAT_MODEL, "chunks_used": len(retrieved_chunks), "response_ms": response_ms}
    }

print("rag_query pipeline ready")

In [ ]:
# ============================================================
# STEP 14: Test the RAG Pipeline
# ============================================================

test_questions = [
    "What is the return policy and how many days do I have?",
    "How can I track my shipment?",
    "I forgot my account password, what should I do?",
    "Do you offer free shipping?",
    "How long does a refund take to process?"
]

print("=" * 65)
for q in test_questions:
    print(f"QUESTION: {q}")
    result = rag_query(q)
    print(f"ANSWER:   {result['answer']}")
    print(f"SOURCE:   {result['sources'][0]['doc_name']} (score: {result['sources'][0]['similarity']})")
    print(f"TIME:     {result['metadata']['response_ms']}ms")
    print("=" * 65)

In [ ]:
# ============================================================
# STEP 15: View MySQL Query Logs (Analytics)
# ============================================================

conn = get_db_connection()
logs = pd.read_sql(
    "SELECT id, LEFT(query_text,55) AS question, top_chunk_id, response_ms, queried_at "
    "FROM query_logs ORDER BY queried_at DESC LIMIT 10",
    conn
)
conn.close()
print("Recent Query Logs from MySQL:")
print(logs.to_string(index=False))

In [ ]:
# ============================================================
# STEP 16: Write FastAPI Application to main.py
# ============================================================

fastapi_app = '''
"""
RAG Contact Center Q&A - FastAPI Application
Run: uvicorn main:app --reload --port 8000
Docs: http://localhost:8000/docs
"""
import time, json
import numpy as np
import faiss
import mysql.connector
from openai import OpenAI
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional

OPENAI_API_KEY   = "sk-your-openai-api-key-here"
EMBEDDING_MODEL  = "text-embedding-ada-002"
CHAT_MODEL       = "gpt-3.5-turbo"
TOP_K            = 3
MYSQL_CONFIG     = {
    "host": "localhost", "port": 3306,
    "user": "root", "password": "your_password",
    "database": "contact_center_rag"
}

client = OpenAI(api_key=OPENAI_API_KEY)
app    = FastAPI(title="RAG Contact Center API", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

print("Loading FAISS index...")
faiss_index = faiss.read_index("faiss_index.bin")
with open("chunks_metadata.json") as f:
    all_chunks = json.load(f)
print(f"Ready: {faiss_index.ntotal} vectors loaded")

class QueryRequest(BaseModel):
    question : str
    category : Optional[str] = None
    top_k    : Optional[int] = TOP_K

class SourceDoc(BaseModel):
    chunk_id: str; doc_name: str; category: str; similarity: float; excerpt: str

class QueryResponse(BaseModel):
    answer: str; sources: List[SourceDoc]; metadata: dict

def get_embedding(text):
    return client.embeddings.create(input=[text.replace("\\n"," ")], model=EMBEDDING_MODEL).data[0].embedding

def search(query, top_k, cat_filter=None):
    qe = np.array([get_embedding(query)], dtype="float32")
    faiss.normalize_L2(qe)
    dists, idxs = faiss_index.search(qe, top_k * 3)
    out = []
    for d, i in zip(dists[0], idxs[0]):
        if i == -1: continue
        c = all_chunks[i].copy()
        if cat_filter and c["category"] != cat_filter: continue
        c["similarity_score"] = float(d)
        out.append(c)
        if len(out) >= top_k: break
    return out

def generate(question, chunks):
    ctx  = "\\n\\n".join([f"[{c[\'doc_name\']}]\\n{c[\'chunk_text\']}" for c in chunks])
    msgs = [
        {"role":"system", "content":"Answer ONLY from the context. If not found, say to contact support."},
        {"role":"user",   "content":f"Context:\\n{ctx}\\n\\nQuestion: {question}\\nAnswer:"}
    ]
    return client.chat.completions.create(model=CHAT_MODEL, messages=msgs,
                                          temperature=0.2, max_tokens=500).choices[0].message.content.strip()

def log(query, chunk_id, answer, ms):
    try:
        conn = mysql.connector.connect(**MYSQL_CONFIG); cur = conn.cursor()
        cur.execute("INSERT INTO query_logs (query_text,top_chunk_id,answer_text,response_ms) VALUES(%s,%s,%s,%s)",
                    (query, chunk_id, answer[:2000], ms))
        conn.commit(); cur.close(); conn.close()
    except: pass

@app.get("/")
def root():
    return {"status": "online", "vectors": faiss_index.ntotal}

@app.post("/query", response_model=QueryResponse)
def query(req: QueryRequest):
    if not req.question.strip(): raise HTTPException(400, "Question empty")
    t      = time.time()
    chunks = search(req.question, req.top_k or TOP_K, req.category)
    if not chunks: raise HTTPException(404, "No relevant documents found")
    answer = generate(req.question, chunks)
    ms     = int((time.time()-t)*1000)
    log(req.question, chunks[0]["chunk_id"], answer, ms)
    return QueryResponse(
        answer=answer,
        sources=[SourceDoc(chunk_id=c["chunk_id"], doc_name=c["doc_name"], category=c["category"],
                           similarity=round(c["similarity_score"],4), excerpt=c["chunk_text"][:200]+"...") for c in chunks],
        metadata={"model":CHAT_MODEL, "chunks":len(chunks), "response_ms":ms}
    )

@app.get("/documents")
def documents():
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cur  = conn.cursor(dictionary=True)
    cur.execute("SELECT * FROM documents")
    docs = cur.fetchall(); cur.close(); conn.close()
    return {"total": len(docs), "documents": docs}

@app.get("/analytics")
def analytics():
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cur  = conn.cursor(dictionary=True)
    cur.execute("SELECT COUNT(*) total, AVG(response_ms) avg_ms FROM query_logs")
    stats = cur.fetchone()
    cur.execute("SELECT LEFT(query_text,60) q, response_ms, queried_at FROM query_logs ORDER BY queried_at DESC LIMIT 5")
    recent = cur.fetchall(); cur.close(); conn.close()
    return {"stats": stats, "recent": recent}
'''

with open('main.py', 'w') as f:
    f.write(fastapi_app)

print("main.py written successfully!")
print("To start the server, open a terminal and run:")
print("  uvicorn main:app --reload --port 8000")
print("Then open: http://localhost:8000/docs")

In [ ]:
# ============================================================
# STEP 17: Test FastAPI Endpoints via requests
# (Run ONLY after starting uvicorn in terminal)
# ============================================================

import requests
BASE = "http://localhost:8000"

try:
    # Health Check
    print("[1] Health Check")
    r = requests.get(f"{BASE}/", timeout=5)
    print(r.json())

    # Query
    print("\n[2] Query Endpoint")
    r = requests.post(f"{BASE}/query", json={"question": "What is the refund process?"}, timeout=30)
    data = r.json()
    print(f"Answer: {data['answer']}")
    print(f"Top source: {data['sources'][0]['doc_name']}")

    # Documents
    print("\n[3] Documents Endpoint")
    r = requests.get(f"{BASE}/documents", timeout=5)
    for doc in r.json()['documents']:
        print(f"  {doc['doc_id']}: {doc['doc_name']}")

    # Analytics
    print("\n[4] Analytics Endpoint")
    r = requests.get(f"{BASE}/analytics", timeout=5)
    print(r.json()['stats'])

except requests.exceptions.ConnectionError:
    print("Server not running. Run: uvicorn main:app --reload --port 8000")

In [ ]:
# ============================================================
# STEP 18: Your Own Question - Interactive Test
# Change the question below and run!
# ============================================================

your_question = "How do I enable two factor authentication?"   # <-- Change this!

result = rag_query(your_question, log_to_db=True)
print(f"Q: {your_question}")
print(f"A: {result['answer']}")
print(f"\nSources:")
for s in result['sources']:
    print(f"  - {s['doc_name']} (score: {s['similarity']})")
print(f"Response time: {result['metadata']['response_ms']}ms")